# LeetCode 1820 — Maximum Number of Accepted Invitations bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-accepted-invitations/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Graph, Matrix (Bipartite Matching)

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def maximumInvitations(self, grid: List[List[int]]) -> int:
        ...
```

Lộ trình: **vét cạn / DP bitmask** (chỉ cho lưới nhỏ) → **tham lam** (và vì sao sai) → **đường tăng Kuhn (thuật toán "Hungarian" dạng DFS)** `O(m · E)` (lời giải nộp) → **Hopcroft–Karp** `O(E √V)` → **luồng cực đại** (Edmonds–Karp trên mạng đơn vị).

## 1. Mục tiêu học

1. Nhận ra bài toán là **ghép cặp cực đại trên đồ thị hai phía** (bipartite matching): bên trái là các bạn nam, bên phải là các bạn nữ, cạnh = "được phép mời".
2. Hiểu khái niệm **đường tăng (augmenting path)** và vì sao "không còn đường tăng ⇔ ghép cặp cực đại" (định lý Berge).
3. Cài đặt **thuật toán Kuhn** bằng DFS với mảng `match` và tập `visited` reset cho **mỗi** bạn nam.
4. Biết các cách nhanh hơn/tổng quát hơn: **Hopcroft–Karp**, **luồng cực đại**.
5. Kiểm chứng bằng **DP bitmask** trên lưới nhỏ ngẫu nhiên có seed; thấy phản ví dụ cho tham lam.

## 2. Đề bài & ví dụ

Có `m` bạn nam và `n` bạn nữ dự tiệc. Cho ma trận `grid` kích thước `m × n` gồm 0/1: `grid[i][j] == 1` nghĩa là bạn nam `i` **có thể mời** bạn nữ `j`. Mỗi bạn nam mời **tối đa một** bạn nữ, mỗi bạn nữ nhận **tối đa một** lời mời.

Trả về **số lời mời được chấp nhận nhiều nhất**.

**Ví dụ 1:** `grid = [[1,1,1],[1,0,1],[0,0,1]]` → `3` (nam 1 → nữ 2, nam 2 → nữ 1, nam 3 → nữ 3; đánh số từ 1).

**Ví dụ 2:** `grid = [[1,0,1,0],[1,0,0,0],[0,0,1,0],[1,1,1,0]]` → `3` (nam 1 → nữ 3, nam 2 → nữ 1, nam 3 không mời ai, nam 4 → nữ 2).

**Ràng buộc:** `1 ≤ m, n ≤ 200`, `grid[i][j] ∈ {0, 1}`.

In [1]:
import random
import sys
import time
from collections import deque
from functools import lru_cache
from typing import List

examples = [
    ([[1, 1, 1], [1, 0, 1], [0, 0, 1]], 3),
    ([[1, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 1, 1, 0]], 3),
]
for g, want in examples:
    print(g, "-> kỳ vọng", want)

def is_valid_matching(grid, pairs):
    """pairs: list (nam, nữ). Hợp lệ nếu mọi cặp là cạnh có thật và không ai bị ghép hai lần."""
    boys = [b for b, _ in pairs]; girls = [g for _, g in pairs]
    return len(set(boys)) == len(boys) and len(set(girls)) == len(girls) and all(grid[b][g] == 1 for b, g in pairs)

[[1, 1, 1], [1, 0, 1], [0, 0, 1]] -> kỳ vọng 3
[[1, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 1, 1, 0]] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Đồ thị hai phía và ghép cặp

Một **ghép cặp** (matching) là tập cạnh không có hai cạnh nào chung đỉnh. Bài hỏi kích thước ghép cặp **lớn nhất**. Danh sách kề của bạn nam `i`: các `j` có `grid[i][j] == 1`.

In [2]:
grid = examples[0][0]
adj = [[j for j, v in enumerate(row) if v] for row in grid]
print("danh sách kề (nam -> các nữ):", adj)
print("ghép [(0,1),(1,0),(2,2)] hợp lệ:", is_valid_matching(grid, [(0, 1), (1, 0), (2, 2)]))
print("ghép [(0,2),(2,2)] hợp lệ:", is_valid_matching(grid, [(0, 2), (2, 2)]))

danh sách kề (nam -> các nữ): [[0, 1, 2], [0, 2], [2]]
ghép [(0,1),(1,0),(2,2)] hợp lệ: True
ghép [(0,2),(2,2)] hợp lệ: False


### 3.2 Đường tăng (augmenting path)

Với một ghép cặp `M`, **đường tăng** là đường đi xen kẽ bắt đầu ở một nam **chưa ghép**, kết thúc ở một nữ **chưa ghép**, các cạnh lần lượt **không thuộc** / **thuộc** `M`. "Lật" đường này (cạnh ngoài `M` vào, cạnh trong `M` ra) làm `|M|` tăng đúng 1.

Ví dụ: `M = {(nam0, nữ0)}`; nam1 chỉ thích nữ0. Đường `nam1 — nữ0 = nam0 — nữ2` (nếu nam0 thích nữ2): lật ⇒ `{(nam1, nữ0), (nam0, nữ2)}`.

In [3]:
M = {(0, 0)}
path = [(1, 0), (0, 0), (0, 2)]          # cạnh: ngoài M, trong M, ngoài M
M2 = (M - {(0, 0)}) | {(1, 0), (0, 2)}
print("trước:", M, "| sau khi lật:", M2)

trước: {(0, 0)} | sau khi lật: {(1, 0), (0, 2)}


### 3.3 Định lý Berge

Một ghép cặp là **cực đại** khi và chỉ khi **không tồn tại** đường tăng. Vì vậy thuật toán: lặp "tìm đường tăng, lật" tới khi không còn.

### 3.4 Đệ quy với `nonlocal` / biến ngoài và tập `visited`

DFS tìm đường tăng dùng mảng `match[j]` (nữ `j` đang ghép với nam nào, `-1` nếu chưa) và tập `vis` các nữ đã thử trong **lần tìm hiện tại**.

In [4]:
match = [-1] * 3
vis = set()
def demo_find(i):
    for j in adj[i]:
        if j in vis: continue
        vis.add(j)
        if match[j] == -1 or demo_find(match[j]):
            match[j] = i
            return True
    return False
for i in range(3):
    vis = set()
    print(f"nam {i}: tìm được? {demo_find(i)} | match (nữ -> nam) = {match}")

nam 0: tìm được? True | match (nữ -> nam) = [0, -1, -1]
nam 1: tìm được? True | match (nữ -> nam) = [1, 0, -1]
nam 2: tìm được? True | match (nữ -> nam) = [1, 0, 2]


### 3.5 Bitmask cho DP trên tập con

Với `n ≤ 10` nữ, tập nữ đã được mời biểu diễn bằng số nguyên `mask`; bit `j` bật ⇔ nữ `j` đã nhận lời. `mask | (1 << j)` thêm nữ `j`, `mask >> j & 1` kiểm tra.

In [5]:
mask = 0
mask |= 1 << 2
print(bin(mask), "nữ 2 đã nhận?", mask >> 2 & 1, "| nữ 0?", mask >> 0 & 1)

0b100 nữ 2 đã nhận? 1 | nữ 0? 0


## 4. Ý tưởng nền tảng / trực giác

Thử tay Ví dụ 1 bằng chiến lược "mỗi nam lần lượt chọn nữ **đầu tiên** còn trống":
- nam 0 → nữ 0; nam 1 → nữ 2 (nữ 0 đã có); nam 2 chỉ thích nữ 2 — **hết chỗ**. Được 2.

Nhưng đáp án là 3! Cách sửa: khi nam 2 muốn nữ 2 (đang ghép với nam 1), ta **hỏi nam 1** có thể chuyển sang nữ khác không. Nam 1 thích {nữ 0, nữ 2}; nữ 0 đang ghép với nam 0 — **hỏi tiếp nam 0**: nam 0 còn nữ 1 trống ⇒ chuyển. Chuỗi "nhường chỗ": `nam2 → nữ2 (của nam1) → nữ0 (của nam0) → nữ1 (trống)`. Đó chính là **đường tăng**.

```
trước:  nam0—nữ0   nam1—nữ2   nam2: (không)
sau:    nam0—nữ1   nam1—nữ0   nam2—nữ2      (3 cặp)
```

## 5. Cách 1 — Brute Force / baseline: DP bitmask trên tập nữ đã nhận lời

**Ý tưởng:** xét lần lượt từng nam `i`; nam `i` hoặc **không mời ai**, hoặc mời một nữ `j` (có cạnh) **chưa nhận lời**. `f(i, mask)` = số cặp lớn nhất từ nam `i` trở đi khi tập nữ đã nhận lời là `mask`. Xét hết mọi khả năng ⇒ chắc chắn đúng (chuẩn đối chiếu).

In [6]:
def max_invites_dp(grid):
    m, n = len(grid), len(grid[0])
    assert n <= 12, "chỉ dùng cho n nhỏ"

    @lru_cache(maxsize=None)
    def f(i, mask):
        if i == m:
            return 0
        best = f(i + 1, mask)                          # nam i không mời ai
        for j in range(n):
            if grid[i][j] and not (mask >> j & 1):
                best = max(best, 1 + f(i + 1, mask | (1 << j)))
        return best

    res = f(0, 0)
    f.cache_clear()
    return res

for g, want in examples:
    print(max_invites_dp(g), "| kỳ vọng", want)

3 | kỳ vọng 3
3 | kỳ vọng 3


**Độ phức tạp:** `O(m · 2ⁿ · n)` thời gian, `O(m · 2ⁿ)` bộ nhớ — với `n = 200` là bất khả. **Nút thắt:** ta nhớ **toàn bộ** tập nữ đã dùng, trong khi lý thuyết đường tăng cho thấy chỉ cần **ghép hiện tại** và khả năng "nhường chỗ".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (Berge):** bắt đầu từ ghép rỗng; lần lượt với mỗi nam `i`, tìm một **đường tăng** xuất phát từ `i`. Nếu có thì lật ⇒ thêm 1 cặp. Không có thì nam `i` sẽ **không bao giờ** cần được xét lại (một định lý của thuật toán Kuhn: nếu từ `i` không có đường tăng ở thời điểm này thì sau khi các lần lật khác xảy ra vẫn không có).

**Hàm `find(i)` (DFS):** với mỗi nữ `j` kề `i`, *nếu* `j` chưa được thử trong lần tìm này:
- đánh dấu `j` đã thử;
- *nếu* `j` còn trống, **hoặc** nam đang giữ `j` (`match[j]`) tìm được chỗ khác (`find(match[j])`) *thì* gán `match[j] = i`, trả `True`.

**Quy tắc quan trọng:** tập `vis` (các nữ đã thử) phải **reset cho mỗi nam** `i` ở vòng ngoài, nhưng **dùng chung** trong cùng một lần `find` (để không lặp vô hạn).

**Invariant:** `match` luôn là một ghép cặp hợp lệ; sau khi xử lý nam `0..i`, `|match|` = kích thước ghép cực đại của đồ thị con gồm các nam `0..i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam (để thấy vì sao cần đường tăng)

In [7]:
def max_invites_greedy(grid):
    taken = set(); cnt = 0
    for row in grid:
        for j, v in enumerate(row):
            if v and j not in taken:
                taken.add(j); cnt += 1
                break
    return cnt

for g, want in examples:
    print("tham lam:", max_invites_greedy(g), "| đúng:", want)

tham lam: 2 | đúng: 3
tham lam: 3 | đúng: 3


Tham lam nhanh (`O(m·n)`) nhưng **sai** vì một quyết định sớm (nam 0 lấy nữ 0) có thể chặn người khác. Thứ tự khác (ví dụ ưu tiên nam ít lựa chọn) cải thiện thực tế nhưng vẫn có phản ví dụ.

### 7.2 Cách 3 — Kuhn (DFS tìm đường tăng) — lời giải chuẩn

In [8]:
class SolutionKuhn:
    def maximumInvitations(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
        match = [-1] * n                       # match[j] = nam đang ghép với nữ j

        def find(i, vis):
            for j in adj[i]:
                if j not in vis:
                    vis.add(j)
                    if match[j] == -1 or find(match[j], vis):
                        match[j] = i
                        return True
            return False

        return sum(find(i, set()) for i in range(m))

for g, want in examples:
    print(SolutionKuhn().maximumInvitations(g), want)

3 3
3 3


**Độ phức tạp:** mỗi lần `find` thăm mỗi nữ tối đa một lần và duyệt mỗi cạnh tối đa một lần ⇒ `O(n + E)`; gọi `m` lần ⇒ `O(m · (n + E))` ≤ `200 · 40000 = 8·10⁶`. Độ sâu đệ quy ≤ `n ≤ 200` — an toàn với giới hạn 1000.

### 7.3 Cách 4 — Hopcroft–Karp (`O(E √V)`)

Thay vì tìm từng đường tăng, mỗi **pha**: BFS từ mọi nam chưa ghép để chia **tầng**, rồi DFS tìm một **tập cực đại các đường tăng ngắn nhất, rời nhau** và lật tất cả. Số pha `O(√V)`.

In [9]:
class SolutionHK:
    def maximumInvitations(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
        INF = float("inf")
        match_l = [-1] * m           # nam -> nữ
        match_r = [-1] * n           # nữ -> nam
        dist = [0] * m

        def bfs():
            q = deque()
            for i in range(m):
                if match_l[i] == -1:
                    dist[i] = 0; q.append(i)
                else:
                    dist[i] = INF
            found = False
            while q:
                i = q.popleft()
                for j in adj[i]:
                    k = match_r[j]
                    if k == -1:
                        found = True
                    elif dist[k] == INF:
                        dist[k] = dist[i] + 1; q.append(k)
            return found

        def dfs(i):
            for j in adj[i]:
                k = match_r[j]
                if k == -1 or (dist[k] == dist[i] + 1 and dfs(k)):
                    match_l[i] = j; match_r[j] = i
                    return True
            dist[i] = INF            # cắt tỉa: nam này không dẫn tới đường tăng trong pha này
            return False

        res = 0
        while bfs():
            for i in range(m):
                if match_l[i] == -1 and dfs(i):
                    res += 1
        return res

for g, want in examples:
    print(SolutionHK().maximumInvitations(g), want)

3 3
3 3


### 7.4 Cách 5 — Luồng cực đại trên mạng đơn vị

Dựng mạng: nguồn `S → nam i` (dung lượng 1), `nam i → nữ j` nếu `grid[i][j] = 1` (dung lượng 1), `nữ j → đích T` (dung lượng 1). Luồng cực đại = ghép cặp cực đại. Dùng Edmonds–Karp (BFS tìm đường tăng trên đồ thị dư). Tổng quát hơn: dễ mở rộng cho "mỗi nữ nhận tối đa `c_j` lời mời" (Bài tập 5).

In [10]:
def max_flow_matching(grid, cap_girl=None):
    m, n = len(grid), len(grid[0])
    S, T = m + n, m + n + 1
    N = m + n + 2
    graph = [[] for _ in range(N)]           # cạnh: [đích, dung lượng còn, chỉ số cạnh ngược]
    def add(u, v, c):
        graph[u].append([v, c, len(graph[v])])
        graph[v].append([u, 0, len(graph[u]) - 1])
    for i in range(m):
        add(S, i, 1)
        for j in range(n):
            if grid[i][j]:
                add(i, m + j, 1)
    for j in range(n):
        add(m + j, T, 1 if cap_girl is None else cap_girl[j])
    flow = 0
    while True:
        par = [None] * N; par[S] = (S, -1)
        q = deque([S])
        while q and par[T] is None:
            u = q.popleft()
            for idx, (v, c, _) in enumerate(graph[u]):
                if c > 0 and par[v] is None:
                    par[v] = (u, idx); q.append(v)
        if par[T] is None:
            return flow
        v = T                                 # mọi dung lượng trên đường đều >= 1, đẩy 1 đơn vị
        while v != S:
            u, idx = par[v]
            e = graph[u][idx]; e[1] -= 1
            graph[v][e[2]][1] += 1
            v = u
        flow += 1

class SolutionFlow:
    def maximumInvitations(self, grid: List[List[int]]) -> int:
        return max_flow_matching(grid)

for g, want in examples:
    print(SolutionFlow().maximumInvitations(g), want)

3 3
3 3


**Độ phức tạp:** mỗi đường tăng tăng luồng 1, tối đa `min(m, n)` lần; mỗi BFS `O(V + E)` ⇒ `O(min(m,n) · (m + n + E))`.

## 8. TRACE từng bước

Trace Kuhn trên Ví dụ 1 (`adj = [[0,1,2],[0,2],[2]]`):

| nam | lời gọi `find` | hành động | `match` (nữ → nam) |
|---|---|---|---|
| 0 | thử nữ 0 (trống) | ghép (0,0) | `[0, -1, -1]` |
| 1 | thử nữ 0 (của nam 0) → `find(0)`: nữ 0 đã thử, nữ 1 trống → nam 0 sang nữ 1 | ghép (1,0) | `[1, 0, -1]` |
| 2 | thử nữ 2 (trống) | ghép (2,2) | `[1, 0, 2]` |

Kết quả 3. (Thứ tự thử khác với minh họa mục 4 nhưng cùng kích thước.)

In [11]:
def trace_kuhn(grid):
    m, n = len(grid), len(grid[0])
    adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    match = [-1] * n
    def find(i, vis, depth):
        pad = "    " * depth
        for j in adj[i]:
            if j in vis:
                print(f"{pad}nam {i}: nữ {j} đã thử trong lần này, bỏ qua"); continue
            vis.add(j)
            if match[j] == -1:
                print(f"{pad}nam {i}: nữ {j} còn trống -> ghép"); match[j] = i; return True
            print(f"{pad}nam {i}: nữ {j} đang ghép với nam {match[j]} -> hỏi nam {match[j]} đổi chỗ")
            if find(match[j], vis, depth + 1):
                print(f"{pad}nam {i}: nhận nữ {j}"); match[j] = i; return True
        print(f"{pad}nam {i}: không tìm được"); return False
    total = 0
    for i in range(m):
        print(f"--- xử lý nam {i}")
        total += find(i, set(), 1)
        print(f"    match = {match}, tổng = {total}")
    return total

trace_kuhn(examples[0][0])

--- xử lý nam 0
    nam 0: nữ 0 còn trống -> ghép
    match = [0, -1, -1], tổng = 1
--- xử lý nam 1
    nam 1: nữ 0 đang ghép với nam 0 -> hỏi nam 0 đổi chỗ
        nam 0: nữ 0 đã thử trong lần này, bỏ qua
        nam 0: nữ 1 còn trống -> ghép
    nam 1: nhận nữ 0
    match = [1, 0, -1], tổng = 2
--- xử lý nam 2
    nam 2: nữ 2 còn trống -> ghép
    match = [1, 0, 2], tổng = 3


3

In [12]:
trace_kuhn(examples[1][0])

--- xử lý nam 0
    nam 0: nữ 0 còn trống -> ghép
    match = [0, -1, -1, -1], tổng = 1
--- xử lý nam 1
    nam 1: nữ 0 đang ghép với nam 0 -> hỏi nam 0 đổi chỗ
        nam 0: nữ 0 đã thử trong lần này, bỏ qua
        nam 0: nữ 2 còn trống -> ghép
    nam 1: nhận nữ 0
    match = [1, -1, 0, -1], tổng = 2
--- xử lý nam 2
    nam 2: nữ 2 đang ghép với nam 0 -> hỏi nam 0 đổi chỗ
        nam 0: nữ 0 đang ghép với nam 1 -> hỏi nam 1 đổi chỗ
            nam 1: nữ 0 đã thử trong lần này, bỏ qua
            nam 1: không tìm được
        nam 0: nữ 2 đã thử trong lần này, bỏ qua
        nam 0: không tìm được
    nam 2: không tìm được
    match = [1, -1, 0, -1], tổng = 2
--- xử lý nam 3
    nam 3: nữ 0 đang ghép với nam 1 -> hỏi nam 1 đổi chỗ
        nam 1: nữ 0 đã thử trong lần này, bỏ qua
        nam 1: không tìm được
    nam 3: nữ 1 còn trống -> ghép
    match = [1, 3, 0, -1], tổng = 3


3

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maximumInvitations(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
        match = [-1] * n                     # match[j]: nam đang ghép với nữ j

        def find(i: int) -> bool:            # tìm đường tăng từ nam i
            for j in adj[i]:
                if j not in vis:
                    vis.add(j)
                    if match[j] == -1 or find(match[j]):
                        match[j] = i
                        return True
            return False

        ans = 0
        for i in range(m):
            vis = set()                      # reset cho MỖI nam
            if find(i):
                ans += 1
        return ans

for g, want in examples:
    assert Solution().maximumInvitations(g) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


(Hàm lồng `find` đọc biến `vis` của hàm ngoài — mỗi vòng `for i` gán `vis` mới, và `find` thấy giá trị mới nhất vì Python tra tên lúc chạy.)

## 10. Vì sao thuật toán đúng

**Định lý Berge:** ghép `M` cực đại ⇔ không có đường tăng. *Chiều ⇒:* nếu có đường tăng thì lật nó được ghép lớn hơn. *Chiều ⇐:* nếu `M*` lớn hơn `M`, hiệu đối xứng `M ⊕ M*` gồm các đường/chu trình xen kẽ; vì `|M*| > |M|` phải có một thành phần chứa nhiều cạnh của `M*` hơn — đó là một đường tăng đối với `M`.

**`find(i)` tìm đúng đường tăng từ `i` (nếu có):** DFS trên đồ thị "nam → nữ (cạnh ngoài M) → nam đang giữ nữ đó (cạnh trong M) → …", dừng khi gặp nữ trống. Tập `vis` bảo đảm mỗi nữ được mở rộng tối đa một lần trong một lần tìm — đủ, vì nếu từ nữ `j` (qua nam `match[j]`) không tới được nữ trống ở lần thử đầu thì các lần sau trong cùng lần tìm cũng không (đồ thị không đổi trong lúc tìm). Khi thành công, các phép gán `match[j] = i` dọc đường đệ quy chính là phép **lật**.

**Xử lý mỗi nam một lần là đủ (bổ đề chuẩn của thuật toán Kuhn):** nếu đối với ghép `M` không có đường tăng xuất phát từ nam `x`, và `M'` nhận được từ `M` bằng cách lật một đường tăng `P`, thì đối với `M'` cũng không có đường tăng từ `x`. *Phác thảo:* giả sử có đường tăng `Q` từ `x` đối với `M'`; nếu `Q` không chạm `P` thì `Q` cũng là đường tăng đối với `M` (mâu thuẫn); nếu chạm, đi theo `Q` tới điểm chạm đầu tiên rồi rẽ theo `P` về phía đầu nữ trống của `P` sẽ được một đường tăng từ `x` đối với `M` (mâu thuẫn). Vì vậy khi kết thúc vòng lặp, không nam nào còn đường tăng ⇒ theo Berge, ghép là cực đại.

Ngoài lập luận, mục 12 đối chiếu với DP bitmask trên hàng nghìn lưới.

## 11. Phân tích Time / Space

Gọi `V = m + n`, `E` = số ô bằng 1 (≤ `m·n = 4·10⁴`).

| Cách | Time | Space |
|---|---|---|
| 1. DP bitmask | `O(m · 2ⁿ · n)` | `O(m · 2ⁿ)` |
| 2. Tham lam (sai) | `O(m · n)` | `O(n)` |
| 3. Kuhn | `O(m · (n + E))` | `O(V + E)` |
| 4. Hopcroft–Karp | `O(E √V)` | `O(V + E)` |
| 5. Luồng (Edmonds–Karp, mạng đơn vị) | `O(min(m,n) · (V + E))` | `O(V + E)` |

In [14]:
def random_grid(m, n, p, rng):
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]

rng = random.Random(1820)
tests = [("200x200 dày p=0.5", random_grid(200, 200, 0.5, rng)),
         ("200x200 thưa p=0.02", random_grid(200, 200, 0.02, rng)),
         ("200x200 tam giác", [[1 if j >= i else 0 for j in range(200)] for i in range(200)])]
for label, g in tests:
    res = []
    for name, sol in [("Kuhn/Nộp", Solution()), ("Hopcroft-Karp", SolutionHK()), ("Flow", SolutionFlow())]:
        t = time.perf_counter(); r = sol.maximumInvitations(g); res.append(r)
        print(f"[{label}] {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1

[200x200 dày p=0.5] Kuhn/Nộp: 200 trong 12 ms
[200x200 dày p=0.5] Hopcroft-Karp: 200 trong 3 ms
[200x200 dày p=0.5] Flow: 200 trong 117 ms
[200x200 thưa p=0.02] Kuhn/Nộp: 196 trong 1 ms
[200x200 thưa p=0.02] Hopcroft-Karp: 196 trong 1 ms


[200x200 thưa p=0.02] Flow: 196 trong 12 ms
[200x200 tam giác] Kuhn/Nộp: 200 trong 1 ms
[200x200 tam giác] Hopcroft-Karp: 200 trong 1 ms


[200x200 tam giác] Flow: 200 trong 53 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu DP bitmask)

In [15]:
sols = [Solution(), SolutionKuhn(), SolutionHK(), SolutionFlow()]
edge = examples + [
    ([[0]], 0),
    ([[1]], 1),
    ([[1, 1, 1]], 1),                    # một nam, nhiều nữ
    ([[1], [1], [1]], 1),                # nhiều nam, một nữ
    ([[1, 0], [0, 1]], 2),
    ([[1, 1], [1, 0]], 2),               # tham lam theo thứ tự sẽ hỏng
    ([[0, 0], [0, 0]], 0),
    ([[1] * 5 for _ in range(5)], 5),
]
for g, want in edge:
    assert max_invites_dp(g) == want
    for s in sols:
        assert s.maximumInvitations(g) == want, (g, s)
    print("OK", g, "->", want)

OK [[1, 1, 1], [1, 0, 1], [0, 0, 1]] -> 3
OK [[1, 0, 1, 0], [1, 0, 0, 0], [0, 0, 1, 0], [1, 1, 1, 0]] -> 3
OK [[0]] -> 0
OK [[1]] -> 1
OK [[1, 1, 1]] -> 1
OK [[1], [1], [1]] -> 1
OK [[1, 0], [0, 1]] -> 2
OK [[1, 1], [1, 0]] -> 2
OK [[0, 0], [0, 0]] -> 0
OK [[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]] -> 5


### 12.2 Lưới ngẫu nhiên seed cố định: so với DP bitmask

In [16]:
rng = random.Random(42)
greedy_wrong = 0
for _ in range(1500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    g = random_grid(m, n, rng.choice([0.2, 0.4, 0.6, 0.9]), rng)
    want = max_invites_dp(g)
    for s in sols:
        assert s.maximumInvitations(g) == want, g
    greedy_wrong += max_invites_greedy(g) != want
print("1500 lưới khớp DP; số lần tham lam sai:", greedy_wrong)

1500 lưới khớp DP; số lần tham lam sai: 107


### 12.3 Kiểm tra ghép cặp trả về là hợp lệ và đạt kích thước

Lấy `match` từ Kuhn, dựng danh sách cặp, kiểm `is_valid_matching` và kích thước.

In [17]:
def kuhn_pairs(grid):
    m, n = len(grid), len(grid[0])
    adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    match = [-1] * n
    def find(i, vis):
        for j in adj[i]:
            if j not in vis:
                vis.add(j)
                if match[j] == -1 or find(match[j], vis):
                    match[j] = i; return True
        return False
    for i in range(m):
        find(i, set())
    return [(match[j], j) for j in range(n) if match[j] != -1]

rng = random.Random(5)
for _ in range(500):
    g = random_grid(rng.randint(1, 8), rng.randint(1, 8), 0.4, rng)
    pairs = kuhn_pairs(g)
    assert is_valid_matching(g, pairs) and len(pairs) == max_invites_dp(g)
print("ghép cặp trả về luôn hợp lệ và cực đại;", kuhn_pairs(examples[0][0]))

ghép cặp trả về luôn hợp lệ và cực đại; [(1, 0), (0, 1), (2, 2)]


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam "chọn nữ đầu tiên còn trống"

In [18]:
g = examples[0][0]
print("tham lam:", max_invites_greedy(g), "| đúng:", Solution().maximumInvitations(g))

tham lam: 2 | đúng: 3


**Vì sao sai:** nam 0 lấy nữ 0, nam 1 lấy nữ 2, nam 2 (chỉ thích nữ 2) hết chỗ. Không có cơ chế "nhường chỗ". **Sửa:** tìm đường tăng (Kuhn).

### Lỗi 2 — Không reset `vis` cho mỗi nam

In [19]:
def wrong_shared_vis(grid):
    m, n = len(grid), len(grid[0])
    adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    match = [-1] * n
    vis = set()                              # SAI: dùng chung cho mọi nam
    def find(i):
        for j in adj[i]:
            if j not in vis:
                vis.add(j)
                if match[j] == -1 or find(match[j]):
                    match[j] = i; return True
        return False
    return sum(find(i) for i in range(m))

g = [[1, 1], [1, 0]]
print("sai:", wrong_shared_vis(g), "| đúng:", Solution().maximumInvitations(g))

sai: 1 | đúng: 2


**Vì sao sai:** nam 0 thử nữ 0 và ghép; nữ 0 bị đánh dấu "đã thử" **vĩnh viễn**. Đến nam 1 (chỉ thích nữ 0), nữ 0 bị bỏ qua nên không thể hỏi nam 0 chuyển sang nữ 1. **Sửa:** `vis = set()` ở đầu mỗi vòng `for i`.

### Lỗi 3 — "Cướp" nữ mà không sắp xếp lại cho người cũ

In [20]:
def wrong_steal(grid):
    m, n = len(grid), len(grid[0])
    match = [-1] * n; ans = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j]:
                match[j] = i                 # SAI: ghi đè, nam cũ mất cặp nhưng vẫn được đếm
                ans += 1
                break
    return ans

g = [[1], [1]]
print("sai:", wrong_steal(g), "| đúng:", Solution().maximumInvitations(g))

sai: 2 | đúng: 1


**Vì sao sai:** chỉ có một nữ nhưng đếm 2 cặp. Khi đổi người ghép với nữ `j`, nam cũ **phải** tìm được chỗ khác (đệ quy `find(match[j])`), nếu không thì không được cướp.

### Lỗi 4 — Không đánh dấu đỉnh đã thăm (kèm một "lỗi giả" hay gặp)

In [21]:
def wrong_vis_boys(grid):
    m, n = len(grid), len(grid[0])
    adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    match = [-1] * n
    def find(i, vis_boys):
        if i in vis_boys:
            return False
        vis_boys.add(i)
        for j in adj[i]:
            if match[j] == -1:
                match[j] = i; return True
        for j in adj[i]:
            if find(match[j], vis_boys):     # vẫn đúng logic, nhưng ...
                match[j] = i; return True
        return False
    return sum(find(i, set()) for i in range(m))

rng = random.Random(0)
mismatch = 0
for _ in range(300):
    g = random_grid(rng.randint(1, 6), rng.randint(1, 6), 0.5, rng)
    mismatch += wrong_vis_boys(g) != max_invites_dp(g)
print("số lưới cho kết quả sai:", mismatch, "(biến thể này thực ra đúng — đánh dấu nam cũng là một cách hợp lệ)")

số lưới cho kết quả sai: 0 (biến thể này thực ra đúng — đánh dấu nam cũng là một cách hợp lệ)


**Ghi chú:** đây là một "lỗi giả" hay gặp trong thảo luận — nhiều bạn tưởng phải đánh dấu nữ mới đúng. Thực ra đánh dấu **nam** đã thăm (mỗi nam mở rộng tối đa một lần) cũng cho thuật toán đúng; chỉ cần bảo đảm **không mở rộng một đỉnh hai lần** trong cùng một lần tìm. Kiểm chứng bằng DP ở trên cho 0 lỗi. Lỗi thật sự là **không đánh dấu gì cả** — dẫn tới đệ quy vô hạn khi đồ thị có chu trình xen kẽ:

In [22]:
def wrong_no_vis(grid, limit=500):
    m, n = len(grid), len(grid[0])
    adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
    match = [-1] * n
    calls = [0]
    def find(i):
        calls[0] += 1
        if calls[0] > limit:
            raise RuntimeError("quá nhiều lời gọi")
        for j in adj[i]:
            if match[j] == -1 or find(match[j]):   # SAI: không đánh dấu
                match[j] = i; return True
        return False
    return sum(find(i) for i in range(m))

try:
    print(wrong_no_vis([[1, 1], [1, 1], [1, 1]]))
except (RuntimeError, RecursionError) as e:
    print("lỗi:", type(e).__name__, "-", e, "| đúng:", Solution().maximumInvitations([[1, 1], [1, 1], [1, 1]]))

lỗi: RuntimeError - quá nhiều lời gọi | đúng: 2


**Vì sao sai:** nam 2 hỏi nam 0 đổi chỗ, nam 0 hỏi nam 1, nam 1 lại hỏi nam 0… vòng lặp không dừng. **Sửa:** tập `vis` (theo nữ hoặc theo nam) cho mỗi lần tìm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi bên tối đa một, tối đa số cặp"** (người–việc, nam–nữ, hàng–cột) ⇒ **ghép cặp hai phía** ⇒ Kuhn (ngắn, đủ nhanh cho `V ≤ vài trăm`), Hopcroft–Karp cho lớn hơn.
- Có **dung lượng** (mỗi nữ nhận `c` lời mời) hoặc nhiều tầng ràng buộc ⇒ **luồng cực đại**.
- Kuhn: `match` theo phía phải, `vis` reset **mỗi** đỉnh trái.
- Định lý König: trong đồ thị hai phía, **ghép cực đại = phủ đỉnh nhỏ nhất** — dùng cho bài "xóa ít hàng/cột nhất".
- Tham lam trên ghép cặp hầu như luôn có phản ví dụ — kiểm bằng DP bitmask.

## 15. Biến thể của bài

1. **Mỗi nữ nhận tối đa `c_j` lời mời** ⇒ luồng với dung lượng `c_j` ở cạnh `nữ j → T`, hoặc nhân bản nữ `j` thành `c_j` bản (Bài tập 5).
2. **Có trọng số** (độ ưa thích), tối đa tổng ⇒ thuật toán Hungarian (Kuhn–Munkres) `O(n³)` — LeetCode 1947 *Maximum Compatibility Score Sum* (bản nhỏ dùng DP bitmask).
3. **Phủ đỉnh nhỏ nhất / tập độc lập lớn nhất** trên đồ thị hai phía (König) — LeetCode 1349 *Maximum Students Taking Exam*.
4. **Ghép cặp trên đồ thị tổng quát** (không hai phía) ⇒ thuật toán Blossom (Edmonds) — khó hơn nhiều.
5. LeetCode 2850, 1066 *Campus Bikes II* (gán có chi phí, DP bitmask).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP bitmask | thử mọi lựa chọn, nhớ tập nữ đã dùng | `O(m 2ⁿ n)` | `O(m 2ⁿ)` | kiểm chứng, `n ≤ 15` |
| 2. Tham lam | nữ đầu tiên còn trống | `O(mn)` | `O(n)` | **sai** — chỉ để so sánh |
| 3. Kuhn | đường tăng bằng DFS | `O(m(n+E))` | `O(V+E)` | **nộp bài** |
| 4. Hopcroft–Karp | nhiều đường tăng ngắn nhất mỗi pha | `O(E√V)` | `O(V+E)` | đồ thị lớn |
| 5. Luồng cực đại | mạng S–nam–nữ–T | `O(min(m,n)(V+E))` | `O(V+E)` | có dung lượng/ràng buộc thêm |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `grid` là ma trận đơn vị `3 × 3`; và `grid = [[1,1,1],[1,1,1]]`?

**Bài 2 (tính tay):** Chạy Kuhn trên `[[1,1],[1,0]]`, ghi `match` sau mỗi nam.

**Bài 3 (điền TODO):** Viết `invitation_list(grid)` trả về list `(nam, nữ)` của một phương án tối ưu, sắp xếp theo nam.

**Bài 4 (tự viết lại):** Viết `min_vertex_cover_size(grid)` — số hàng + cột ít nhất cần chọn để mọi ô `1` nằm trên một hàng hoặc cột đã chọn — và kiểm định lý König bằng vét cạn.

**Bài 5 (mini challenge):** Mỗi nữ `j` nhận tối đa `cap[j]` lời mời (mỗi nam vẫn mời tối đa một). Viết `max_invites_capacity(grid, cap)` và đối chiếu với vét cạn.

In [23]:
def invitation_list(grid):
    # TODO
    pass

def min_vertex_cover_size(grid):
    # TODO: định lý König
    pass

def max_invites_capacity(grid, cap):
    # TODO: nhân bản nữ hoặc luồng
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Ma trận đơn vị: mỗi nam một nữ riêng | 3 | Hai nam, ba nữ đều được: `min(2, 3) = 2` |
| 2 | Nam 0 lấy nữ 0 | Nam 1 thích nữ 0 → hỏi nam 0 | Nam 0 chuyển sang nữ 1: `match = [1, 0]` |
| 3 | Dùng `match` của Kuhn | Đảo thành cặp `(match[j], j)` | Sắp xếp theo nam |
| 4 | König: kích thước phủ đỉnh = ghép cực đại | Vét cạn: thử mọi tập hàng, phần còn lại các cột phải phủ | Với mỗi tập hàng `R`, cần chọn mọi cột có ô 1 ở hàng ngoài `R` |
| 5 | Nhân bản nữ `j` thành `cap[j]` cột | Hoặc dùng `max_flow_matching(grid, cap)` | Vét cạn: DP với vector số lời mời còn lại của mỗi nữ |

## 19. Đáp án tham khảo

In [24]:
r1 = [Solution().maximumInvitations(g) for g in ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], [[1, 1, 1], [1, 1, 1]])]
print("Bài 1:", r1); assert r1 == [3, 2]
print("Bài 2:"); assert trace_kuhn([[1, 1], [1, 0]]) == 2

Bài 1: [3, 2]
Bài 2:
--- xử lý nam 0
    nam 0: nữ 0 còn trống -> ghép
    match = [0, -1], tổng = 1
--- xử lý nam 1
    nam 1: nữ 0 đang ghép với nam 0 -> hỏi nam 0 đổi chỗ
        nam 0: nữ 0 đã thử trong lần này, bỏ qua
        nam 0: nữ 1 còn trống -> ghép
    nam 1: nhận nữ 0
    match = [1, 0], tổng = 2


In [25]:
def invitation_list(grid):
    return sorted(kuhn_pairs(grid))

def min_vertex_cover_size(grid):
    return Solution().maximumInvitations(grid)            # định lý König

def min_vertex_cover_brute(grid):
    m, n = len(grid), len(grid[0]); best = m + n
    for R in range(1 << m):
        cols = set()
        for i in range(m):
            if not (R >> i & 1):
                cols.update(j for j in range(n) if grid[i][j])
        best = min(best, bin(R).count("1") + len(cols))
    return best

rng = random.Random(12)
for _ in range(400):
    g = random_grid(rng.randint(1, 6), rng.randint(1, 6), 0.45, rng)
    pairs = invitation_list(g)
    assert is_valid_matching(g, pairs) and len(pairs) == max_invites_dp(g)
    assert min_vertex_cover_size(g) == min_vertex_cover_brute(g)
print("Bài 3, 4: OK", invitation_list(examples[1][0]))

Bài 3, 4: OK [(0, 2), (1, 0), (3, 1)]


In [26]:
def max_invites_capacity(grid, cap):
    return max_flow_matching(grid, cap_girl=cap)

def capacity_brute(grid, cap):
    m, n = len(grid), len(grid[0])
    @lru_cache(maxsize=None)
    def f(i, left):
        if i == m:
            return 0
        best = f(i + 1, left)
        for j in range(n):
            if grid[i][j] and left[j] > 0:
                nl = list(left); nl[j] -= 1
                best = max(best, 1 + f(i + 1, tuple(nl)))
        return best
    return f(0, tuple(cap))

def capacity_by_cloning(grid, cap):
    cols = [j for j in range(len(grid[0])) for _ in range(cap[j])]
    if not cols:
        return 0
    big = [[row[j] for j in cols] for row in grid]
    return Solution().maximumInvitations(big)

rng = random.Random(21)
for _ in range(400):
    m, n = rng.randint(1, 6), rng.randint(1, 4)
    g = random_grid(m, n, 0.5, rng)
    cap = [rng.randint(0, 3) for _ in range(n)]
    want = capacity_brute(g, cap)
    assert max_invites_capacity(g, cap) == want == capacity_by_cloning(g, cap), (g, cap)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
adj = [[j for j in range(n) if grid[i][j]] for i in range(m)]
match = [-1] * n                       # nữ -> nam
def find(i):
    for j in adj[i]:
        if j not in vis:
            vis.add(j)
            if match[j] == -1 or find(match[j]):
                match[j] = i
                return True
    return False
ans = 0
for i in range(m):
    vis = set()                        # reset mỗi nam
    ans += find(i)
```

**Checklist:**
- [ ] Nhận ra **ghép cặp hai phía**, không dùng tham lam?
- [ ] `vis` reset cho **mỗi** nam, dùng chung trong cùng một lần `find`?
- [ ] Khi "cướp" nữ, nam cũ phải tìm được chỗ khác (`find(match[j])`)?
- [ ] Độ sâu đệ quy ≤ `n ≤ 200` (an toàn); lưới lớn hơn thì cân nhắc Hopcroft–Karp/không đệ quy.
- [ ] Lưới toàn 0 trả 0.